In [1]:
import json
import pandas as pd

raw_path = "../data/raw/fever/shared_task_dev.jsonl"

records = []

with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Records:", len(records))

Records: 19998


In [2]:
fever_records = [
    r for r in records
    if r["label"] in ["SUPPORTS", "REFUTES"]
]

print("Usable records:", len(fever_records))

Usable records: 13332


In [3]:
evidence_refs = []

for record in fever_records:

    for evidence_set in record["evidence"]:

        for item in evidence_set:

            if len(item) < 4:
                continue

            annotation_id = item[0]
            evidence_id = item[1]
            page_title = item[2]
            sentence_id = item[3]

            if (
                page_title is not None
                and sentence_id is not None
            ):
                evidence_refs.append({
                    "record_id": record["id"],
                    "page_title": page_title,
                    "sentence_id": sentence_id
                })

print("Evidence references:", len(evidence_refs))

Evidence references: 28625


In [4]:
pd.DataFrame(evidence_refs).head(10)

,record_id,page_title,sentence_id
0,137334,Soul_Food_-LRB-film-RRB-,0
1,137334,Soul_Food_-LRB-film-RRB-,0
2,137334,Soul_Food_-LRB-film-RRB-,0
3,137334,Soul_Food_-LRB-film-RRB-,0
4,137334,Soul_Food_-LRB-film-RRB-,0
5,111897,Telemundo,0
6,111897,Telemundo,1
7,111897,Telemundo,4
8,111897,Hispanic_and_Latino_Americans,0
9,111897,Telemundo,5


In [5]:
evidence_df = pd.DataFrame(evidence_refs)

unique_pages = (
    evidence_df["page_title"]
    .dropna()
    .unique()
)

print("Unique Wikipedia pages needed:", len(unique_pages))

Unique Wikipedia pages needed: 2892


In [6]:
import requests
import time
import json
from pathlib import Path
from urllib.parse import quote

# Output directory
output_dir = Path("../data/external/fever_evidence")
output_dir.mkdir(parents=True, exist_ok=True)

cache_path = output_dir / "wikipedia_pages.json"

# Your unique page titles
pages = list(unique_pages)

print("Pages to fetch:", len(pages))

Pages to fetch: 2892


In [7]:
API_URL = "https://en.wikipedia.org/w/api.php"

session = requests.Session()

session.headers.update({
    "User-Agent": "PredictiveHallucinationEstimator/1.0 "
                  "(research project; educational use)"
})


def fetch_page_batch(titles):
    """
    Fetch multiple Wikipedia pages in one API request.
    """

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": 1,
        "titles": "|".join(titles),
        "redirects": 1,
    }

    response = session.get(
        API_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [8]:
test_pages = pages[:10]

result = fetch_page_batch(test_pages)

print(result.keys())

dict_keys(['continue', 'warnings', 'query'])


In [9]:
for page_id, page in result["query"]["pages"].items():
    print("\nTITLE:", page.get("title"))
    print("EXTRACT:", page.get("extract", "")[:300])


TITLE: Soul Food -LRB-film-RRB-
EXTRACT: 

TITLE: Savages -LRB-2012 film-RRB-
EXTRACT: 

TITLE: Andrew Kevin Walker
EXTRACT: Andrew Kevin Walker (born August 14, 1964) is an American screenwriter. He is known for having written Seven (1995), for which he earned a nomination for the BAFTA Award for Best Original Screenplay, as well as several other films, including 8mm (1999), Sleepy Hollow (1999) and many uncredited scrip

TITLE: Cretaceous
EXTRACT: 

TITLE: Damon Albarn
EXTRACT: 

TITLE: Hispanic and Latino Americans
EXTRACT: 

TITLE: Mogadishu
EXTRACT: 

TITLE: Murda Beatz
EXTRACT: 

TITLE: Nicholas Brody
EXTRACT: 

TITLE: Telemundo
EXTRACT: 


In [10]:
all_pages = {}

batch_size = 20

for start in range(0, len(pages), batch_size):

    batch = pages[start:start + batch_size]

    try:
        result = fetch_page_batch(batch)

        for page_id, page in result["query"]["pages"].items():

            title = page.get("title")

            if title:
                all_pages[title] = {
                    "page_id": page_id,
                    "extract": page.get("extract", "")
                }

        print(
            f"Fetched {min(start + batch_size, len(pages))}"
            f"/{len(pages)}"
        )

        time.sleep(0.2)

    except Exception as e:

        print(
            f"Error around batch {start}: {e}"
        )

        time.sleep(2)

Fetched 20/2892
Fetched 40/2892
Fetched 60/2892
Fetched 80/2892
Fetched 100/2892
Fetched 120/2892
Fetched 140/2892
Fetched 160/2892
Fetched 180/2892
Error around batch 180: 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&format=json&prop=extracts&explaintext=1&titles=Hanford_Site%7CKentucky%7CKelly_Preston%7CKesha%7CVeeru_Devgan%7CStar_Trek-COLON-_Discovery%7CFrench_Montana%7CFloppy_disk%7CHorse%7CForceps%7CThe_Closer%7CMud_-LRB-2012_film-RRB-%7CMatthew_McConaughey%7CRobert_Palmer_-LRB-writer-RRB-%7CMen_in_Black_II%7CMount_Rushmore%7CThe_Concert_for_Bangladesh%7CAvenged_Sevenfold_-LRB-album-RRB-%7CMani_Ratnam%7CLincoln%E2%80%93Douglas_debates&redirects=1
Error around batch 200: 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&format=json&prop=extracts&explaintext=1&titles=Easy_A%7CCamden%2C_New_Jersey%7CCity%7CBret_Easton_Ellis%7CQuincy%2C_Illinois%7CHomer_Hickam%7CCheese_in_the_Trap_-LRB-TV_series

In [11]:
with open(
    cache_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_pages,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", cache_path)
print("Pages retrieved:", len(all_pages))

Saved: ../data/external/fever_evidence/wikipedia_pages.json
Pages retrieved: 1151


In [12]:
missing_pages = [
    page
    for page in pages
    if page not in all_pages
]

print(
    "Requested pages:",
    len(pages)
)

print(
    "Retrieved pages:",
    len(all_pages)
)

print(
    "Missing pages:",
    len(missing_pages)
)

print(
    "First missing pages:",
    missing_pages[:20]
)

Requested pages: 2892
Retrieved pages: 1151
Missing pages: 2730
First missing pages: ['Soul_Food_-LRB-film-RRB-', 'Hispanic_and_Latino_Americans', 'Damon_Albarn', 'Savages_-LRB-2012_film-RRB-', 'Andrew_Kevin_Walker', 'Murda_Beatz', 'Nicholas_Brody', 'Carrie_Mathison', 'Charles_Manson', 'Sean_Penn', 'Brad_Wilk', 'The_Millers', 'Edgar_Wright', 'Ann_Richards', 'Drake_Bell', 'Janet_Leigh', 'Simón_Bolívar', 'Bermuda_Triangle', 'Hot_Right_Now', 'Giada_at_Home']


In [13]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [14]:
sentence_records = []

for page_title, page_data in all_pages.items():

    text = page_data["extract"]

    if not text:
        continue

    doc = nlp(text)

    for sentence_id, sent in enumerate(doc.sents):

        sentence_records.append({
            "page_title": page_title,
            "sentence_id": sentence_id,
            "sentence": sent.text.strip()
        })

sentence_df = pd.DataFrame(
    sentence_records
)

print(
    "Sentence rows:",
    len(sentence_df)
)

print(
    sentence_df.head()
)

Sentence rows: 9780
            page_title  sentence_id  \
0  Andrew Kevin Walker            0   
1  Andrew Kevin Walker            1   
2  Andrew Kevin Walker            2   
3  Andrew Kevin Walker            3   
4  Andrew Kevin Walker            4   

                                            sentence  
0  Andrew Kevin Walker (born August 14, 1964) is ...  
1  He is known for having written Seven (1995), f...  
2  == Early life and education ==\nWalker was bor...  
3  During his childhood, he moved to Mechanicsbur...  
4  He attended the Mechanicsburg Area Senior High...  
